# Structural Tracking vs Column Tracking

DataFerret has two related but distinct tracking mechanisms:

## Column Tracking
Tracks which specific columns you read from a DataFrame.
- If you read `df['a']`, only changes to column 'a' are tracked
- Adding new columns is OK (LEQ mode)
- Deleting columns you read is a violation

## Structural Tracking  
Tracks when you **explicitly** read the DataFrame's *structure* itself.
- If you read `df.columns`, any column changes affect you
- If you read `len(df)`, row count changes affect you

**Important**: Structure-using operations (like `df['x'] = 3`) do NOT count as structural reads,
even though they internally access structure. Only explicit user access is tracked.

In [ ]:
import pandas as pd

%structural_tracking enforce

## Column Tracking Only

When you only access specific column values, adding new columns is fine.

In [ ]:
# Create DataFrame
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})

In [ ]:
# Only read column 'a' values - no structural read
values = df['a'].sum()
print(f"Sum of column a: {values}")

In [ ]:
# Adding column 'c' is OK - we didn't read the column structure
df['c'] = [7, 8, 9]
print("Added column c - no violation!")

## With Structural Tracking

When you **explicitly** access `.columns`, `.shape`, etc., structure changes are tracked.

In [ ]:
# Create fresh DataFrame
df2 = pd.DataFrame({'x': [1, 2], 'y': [3, 4]})

In [ ]:
# EXPLICIT structural read - this IS tracked
cols = df2.columns
print(f"Columns: {list(cols)}")

In [ ]:
# Adding column 'z' after reading .columns would be flagged
# in SDC if this cell comes after the read
df2['z'] = [5, 6]
print(f"Added column z")

## Key Distinction: Using vs Revealing

Setting a column (`df['x'] = value`) does NOT record structural reads, even though
it internally accesses `.columns` and `.index` for broadcasting.

In [ ]:
# Fresh DataFrame
df3 = pd.DataFrame({'a': [1, 2, 3]})

In [ ]:
# These are structure-USING operations - NOT tracked as structural reads
df3['b'] = [4, 5, 6]        # __setitem__
_ = df3['a']                 # __getitem__
_ = df3 + 10                 # arithmetic
_ = df3.mean()               # aggregation
print("All structure-using operations - no structural reads recorded!")

In [ ]:
# These are structure-REVEALING operations - TRACKED
print(f"Columns (TRACKED): {list(df3.columns)}")
print(f"Shape (TRACKED): {df3.shape}")
print(f"Len (TRACKED): {len(df3)}")

## Combined Example

Reading both column values AND column structure.

In [ ]:
df4 = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})

In [ ]:
# Read both column values AND column structure
print(f"Columns: {list(df4.columns)}")  # Structural read (explicit)
print(f"Sum of a: {df4['a'].sum()}")     # Column read (no structural read)

In [ ]:
# Both of these would be tracked:
# - Modifying column 'a' (column tracking)
# - Adding new columns (structural tracking, because we read .columns)

In [ ]:
%structural_tracking warn